verify the environment before touching anything

In [1]:
import os, zipfile, yaml
from pathlib import Path

# Locate the dataset
DETECTION_ROOT = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'data.yaml' in files:
        DETECTION_ROOT = root
        break

print("Detected dataset root:", DETECTION_ROOT)
assert DETECTION_ROOT is not None, "data.yaml not found — is the dataset attached as input?"

with open(Path(DETECTION_ROOT) / 'data.yaml') as f:
    data_yaml = yaml.safe_load(f)
print("\ndata.yaml contents:")
print(data_yaml)

print("\nSplit inventory:")
for s in ['train', 'val', 'test']:
    labels_dir = Path(DETECTION_ROOT) / s / 'labels'
    n_labels = len(list(labels_dir.glob('*.txt'))) if labels_dir.exists() else 'labels dir not found'

    zip_path = Path(DETECTION_ROOT) / f'{s}_images.zip'
    extracted_dir = Path(DETECTION_ROOT) / f'{s}_images'
    if zip_path.exists():
        with zipfile.ZipFile(zip_path) as zf:
            n_images = len([n for n in zf.namelist() if n.lower().endswith('.jpg')])
        img_source = f"zip ({zip_path.name})"
    elif extracted_dir.exists():
        n_images = len(list(extracted_dir.glob('*.jpg')))
        img_source = f"extracted folder ({extracted_dir.name})"
    else:
        n_images, img_source = 'not found', 'NEITHER zip nor extracted folder found'

    print(f"  {s}: {n_images} images ({img_source}), {n_labels} labels")

Detected dataset root: /kaggle/input/datasets/redwanahmed2025/dental-detection-final-v3/dental_final

data.yaml contents:
{'names': ['Cavities', 'Damage', 'Infection', 'Wisdom'], 'nc': 4, 'test': '/kaggle/working/dental_final/test/images', 'train': '/kaggle/working/dental_final/train/images', 'val': '/kaggle/working/dental_final/val/images'}

Split inventory:
  train: 7998 images (extracted folder (train_images)), 7998 labels
  val: 993 images (extracted folder (val_images)), 993 labels
  test: 1009 images (extracted folder (test_images)), 1009 labels


In [2]:
import pandas as pd

records = []
for split in ['train', 'val', 'test']:
    labels_dir = Path(DETECTION_ROOT) / split / 'labels'
    images_dir = Path(DETECTION_ROOT) / f'{split}_images'

    for label_path in sorted(labels_dir.glob('*.txt')):
        image_id = label_path.stem
        image_path = images_dir / f'{image_id}.jpg'

        classes_present = set()
        if label_path.stat().st_size > 0:
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        classes_present.add(int(parts[0]))

        records.append({
            'image_id': image_id,
            'image_path': str(image_path),
            'label_path': str(label_path),
            'original_split': split,          # kept for our own bookkeeping only —
                                                # the new split will ignore this column
            'has_cavities': int(0 in classes_present),
            'has_damage': int(1 in classes_present),
            'has_infection': int(2 in classes_present),
            'has_wisdom': int(3 in classes_present),
        })

master_index = pd.DataFrame(records)

print(f"Total images indexed: {len(master_index)}")
print(f"\nBy original Part A split (for reference only, not used going forward):")
print(master_index['original_split'].value_counts())

print(f"\nImages containing each class (at least one instance):")
for cls in ['has_cavities', 'has_damage', 'has_infection', 'has_wisdom']:
    print(f"  {cls}: {master_index[cls].sum()}")

label_cols = ['has_cavities', 'has_damage', 'has_infection', 'has_wisdom']
n_empty = len(master_index) - master_index[label_cols].any(axis=1).sum()
print(f"\nImages with no labelled instances of any class: {n_empty}")

missing = master_index[~master_index['image_path'].apply(lambda p: Path(p).exists())]
print(f"Label files with no matching image file: {len(missing)}")
if len(missing) > 0:
    print(missing[['image_id', 'original_split']].head())

master_index.to_csv('/kaggle/working/master_index.csv', index=False)
print("\nSaved: /kaggle/working/master_index.csv")

Total images indexed: 10000

By original Part A split (for reference only, not used going forward):
original_split
train    7998
test     1009
val       993
Name: count, dtype: int64

Images containing each class (at least one instance):
  has_cavities: 3151
  has_damage: 3542
  has_infection: 2520
  has_wisdom: 2378

Images with no labelled instances of any class: 177
Label files with no matching image file: 0

Saved: /kaggle/working/master_index.csv


Step 3 — Stratified split into Test / Validation / SSL Pool (seeded)

In [3]:
!pip install iterative-stratification -q

import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

SEED = 42  # LOCK THIS IN — every notebook from here on uses this exact value

label_cols = ['has_cavities', 'has_damage', 'has_infection', 'has_wisdom']
X_dummy = np.zeros((len(master_index), 1))   # stratifier only looks at Y; X is a placeholder
Y = master_index[label_cols].values

mskf = MultilabelStratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

fold_assignment = np.empty(len(master_index), dtype=int)
for fold_id, (_, fold_idx) in enumerate(mskf.split(X_dummy, Y)):
    fold_assignment[fold_idx] = fold_id
master_index['fold'] = fold_assignment

master_index['role'] = 'pool'
master_index.loc[master_index['fold'] == 0, 'role'] = 'test'
master_index.loc[master_index['fold'] == 1, 'role'] = 'val'

pool_fold_map = {orig_fold: i for i, orig_fold in enumerate(range(2, 10))}  # folds 2-9 -> pool_fold 0-7
master_index['pool_fold'] = master_index['fold'].map(pool_fold_map)

print("Role counts:")
print(master_index['role'].value_counts())

print("\nPer-class positive rate by role (should be close across all three):")
print(master_index.groupby('role')[label_cols].mean().round(3))

test_ids = set(master_index.loc[master_index['role']=='test', 'image_id'])
val_ids  = set(master_index.loc[master_index['role']=='val', 'image_id'])
pool_ids = set(master_index.loc[master_index['role']=='pool', 'image_id'])
print("\nDisjointness (all must read 0):")
print("  test ∩ pool:", len(test_ids & pool_ids))
print("  val ∩ pool: ", len(val_ids & pool_ids))
print("  test ∩ val: ", len(test_ids & val_ids))

print("\nPool sub-fold sizes (should each be ~1000 — these are your ρ=0.10 building blocks):")
print(master_index.loc[master_index['role']=='pool', 'pool_fold'].value_counts().sort_index())

Role counts:
role
pool    8000
test    1000
val     1000
Name: count, dtype: int64

Per-class positive rate by role (should be close across all three):
      has_cavities  has_damage  has_infection  has_wisdom
role                                                     
pool         0.315       0.354          0.252       0.238
test         0.315       0.354          0.252       0.238
val          0.315       0.354          0.252       0.238

Disjointness (all must read 0):
  test ∩ pool: 0
  val ∩ pool:  0
  test ∩ val:  0

Pool sub-fold sizes (should each be ~1000 — these are your ρ=0.10 building blocks):
pool_fold
0.0    1000
1.0    1000
2.0    1000
3.0    1000
4.0    1000
5.0    1000
6.0    1000
7.0    1000
Name: count, dtype: int64


Step 4 — Build the ρ=0.20 labelled subset, finalize, and publish

In [4]:
master_index['is_labelled_subset'] = (
    (master_index['role'] == 'pool') & (master_index['pool_fold'] <= 1)
).astype(int)

print("Labelled fine-tuning subset (rho=0.20) size:", master_index['is_labelled_subset'].sum())
print("As fraction of full dataset:", master_index['is_labelled_subset'].mean().round(3))

print("\nPer-class positive rate, labelled subset vs full pool (should be close):")
comparison = pd.DataFrame({
    'labelled_subset': master_index.loc[master_index['is_labelled_subset']==1, label_cols].mean(),
    'full_pool': master_index.loc[master_index['role']=='pool', label_cols].mean(),
})
print(comparison.round(3))

final_cols = ['image_id', 'image_path', 'label_path', 'role', 'pool_fold',
              'is_labelled_subset', 'has_cavities', 'has_damage', 'has_infection', 'has_wisdom']
partition = master_index[final_cols]
partition.to_csv('/kaggle/working/partition.csv', index=False)

print("\nFinal partition summary:")
print(f"  test: {(partition.role=='test').sum()}")
print(f"  val: {(partition.role=='val').sum()}")
print(f"  pool (all, unlabelled input to SSL): {(partition.role=='pool').sum()}")
print(f"  labelled subset (rho=0.20, inside pool): {partition.is_labelled_subset.sum()}")
print(f"  unlabelled-only remainder: {((partition.role=='pool') & (partition.is_labelled_subset==0)).sum()}")
print(f"\nSaved: /kaggle/working/partition.csv")
print(f"SEED USED: {SEED} — must match in every downstream notebook")

Labelled fine-tuning subset (rho=0.20) size: 2000
As fraction of full dataset: 0.2

Per-class positive rate, labelled subset vs full pool (should be close):
               labelled_subset  full_pool
has_cavities             0.315      0.315
has_damage               0.354      0.354
has_infection            0.252      0.252
has_wisdom               0.238      0.238

Final partition summary:
  test: 1000
  val: 1000
  pool (all, unlabelled input to SSL): 8000
  labelled subset (rho=0.20, inside pool): 2000
  unlabelled-only remainder: 6000

Saved: /kaggle/working/partition.csv
SEED USED: 42 — must match in every downstream notebook


Step 5 — Build clean eval folders, get the supervised reference number

In [5]:
### Step 5a — build proper images/ + labels/ folders for val and test only
import shutil

EVAL_DIR = Path('/kaggle/working/eval')

for split_name in ['val', 'test']:
    img_out = EVAL_DIR / split_name / 'images'
    lbl_out = EVAL_DIR / split_name / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    rows = partition[partition['role'] == split_name]
    for _, row in rows.iterrows():
        shutil.copy2(row['image_path'], img_out / f"{row['image_id']}.jpg")
        shutil.copy2(row['label_path'], lbl_out / f"{row['image_id']}.txt")

    print(f"{split_name}: {len(list(img_out.glob('*.jpg')))} images, {len(list(lbl_out.glob('*.txt')))} labels copied")

eval_yaml = {
    'train': str(EVAL_DIR / 'val' / 'images'),  # unused for eval-only, but Ultralytics expects the key present
    'val': str(EVAL_DIR / 'val' / 'images'),
    'test': str(EVAL_DIR / 'test' / 'images'),
    'nc': 4,
    'names': ['Cavities', 'Damage', 'Infection', 'Wisdom'],
}
with open(EVAL_DIR / 'eval_data.yaml', 'w') as f:
    yaml.dump(eval_yaml, f)
print("\nSaved:", EVAL_DIR / 'eval_data.yaml')

val: 1000 images, 1000 labels copied
test: 1000 images, 1000 labels copied

Saved: /kaggle/working/eval/eval_data.yaml


In [6]:
### Step 5b — evaluate YOLOv12 on the new test split
!pip install ultralytics -q
from ultralytics import YOLO

yolov12_ckpt = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'best.pt' in files and 'yolov12' in root.lower():
        yolov12_ckpt = os.path.join(root, 'best.pt')
        break
print("YOLOv12 checkpoint:", yolov12_ckpt)
assert yolov12_ckpt is not None, "Not found — is yolov12-checkpoint attached as input?"

model = YOLO(yolov12_ckpt)
reference_metrics = model.val(
    data=str(EVAL_DIR / 'eval_data.yaml'),
    split='test',
    imgsz=640,
    batch=16,
    iou=0.6,
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLOv12 checkpoint: /kaggle/input/datasets/farhaanjumnisha/yolov12-checkpoint/yolov12m_dental/dry_run/weights/best.pt
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv12m summary (fused): 169 layers, 20,107,996 parameters, 0 gradients, 70.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1243.0±377.9 MB/s, size: 36.3 KB)
val: Scanning /kaggle/working/eval/test/labels... 1000 images, 31 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000 1.3Kit/s 0.8s
val: New cache created: /kag